# Network Intrusion Detection System using UNSW-NB15
This notebook is prepared for **ICT4416 AI in Cybersecurity – IA4**.

It covers:
- Exploratory Data Analysis (EDA)
- Data preprocessing
- Multiple ML models
- A basic Deep Neural Network
- Evaluation using Accuracy, Precision, Recall, F2, F2-Macro, PR-AUC
- Confusion matrices and conclusions

## Notes
- Put `UNSW_NB15_train_40k.csv` and `UNSW_NB15_test_10k.csv` in the same folder as this notebook.
- The assignment PDF asks for binary classification of network traffic as **Normal (0)** or **Attack (1)**.
- F2 score is included because it gives more importance to recall, which is critical in intrusion detection.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    fbeta_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
    PrecisionRecallDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

In [ ]:
TRAIN_PATH = "UNSW_NB15_train_40k.csv"
TEST_PATH = "UNSW_NB15_test_10k.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

TARGET = "label"
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]
X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = [c for c in X_train.columns if c not in categorical_cols]

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nCategorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)
print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True).round(4))
print("\nTest class distribution:")
print(y_test.value_counts(normalize=True).round(4))
print("\nMissing values in train:")
print(train_df.isnull().sum())

## EDA

In [ ]:
os.makedirs("outputs", exist_ok=True)

plt.figure(figsize=(6, 4))
train_df[TARGET].value_counts().sort_index().plot(kind="bar")
plt.title("Training Set Class Distribution")
plt.xlabel("Label (0=Normal, 1=Attack)")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("outputs/class_distribution.png")
plt.show()

In [ ]:
for col in categorical_cols:
    plt.figure(figsize=(8, 4))
    train_df[col].value_counts().head(15).plot(kind="bar")
    plt.title(f"Top Categories in {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(f"outputs/{col}_distribution.png")
    plt.show()

In [ ]:
for col in numerical_cols:
    plt.figure(figsize=(7, 4))
    plt.hist(train_df[col], bins=30)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(f"outputs/{col}_hist.png")
    plt.show()

In [ ]:
grouped_means = train_df.groupby(TARGET)[numerical_cols].mean().T
grouped_means.columns = ["Normal (0)", "Attack (1)"]
grouped_means.round(3)

## Preprocessing
- Median imputation for numerical columns
- Most-frequent imputation for categorical columns
- IQR-based outlier capping
- One-hot encoding for `proto`, `state`, `service`
- Standard scaling for models that depend on feature magnitude

In [ ]:
class IQRClipper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.columns_ = X.columns.tolist()
        self.bounds_ = {}
        for col in self.columns_:
            q1 = X[col].quantile(0.25)
            q3 = X[col].quantile(0.75)
            iqr = q3 - q1
            self.bounds_[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columns_).copy()
        for col in self.columns_:
            lower, upper = self.bounds_[col]
            X[col] = X[col].clip(lower=lower, upper=upper)
        return X

def build_preprocessor(scale_numeric=True, dense_output=False, include_categorical=True):
    num_pipeline_steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("clipper", IQRClipper()),
    ]
    if scale_numeric:
        num_pipeline_steps.append(("scaler", StandardScaler()))

    transformers = [
        ("num", Pipeline(num_pipeline_steps), numerical_cols)
    ]

    if include_categorical:
        cat_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=not dense_output)),
        ])
        transformers.append(("cat", cat_pipeline, categorical_cols))

    return ColumnTransformer(
        transformers=transformers,
        sparse_threshold=0.0 if dense_output else 0.3
    )

## Model Building

In [ ]:
models = {
    "Logistic Regression": {
        "model": LogisticRegression(max_iter=1500, class_weight="balanced"),
        "preprocessor_args": dict(scale_numeric=True, dense_output=False, include_categorical=True),
    },
    "Naive Bayes": {
        "model": GaussianNB(),
        "preprocessor_args": dict(scale_numeric=True, dense_output=True, include_categorical=True),
    },
    "KNN": {
        "model": KNeighborsClassifier(n_neighbors=9),
        "preprocessor_args": dict(scale_numeric=True, dense_output=False, include_categorical=True),
    },
    "SVM": {
        "model": LinearSVC(class_weight="balanced", max_iter=4000),
        "preprocessor_args": dict(scale_numeric=True, dense_output=False, include_categorical=True),
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(max_depth=10, class_weight="balanced", random_state=42),
        "preprocessor_args": dict(scale_numeric=False, dense_output=False, include_categorical=True),
    },
    "Random Forest": {
        "model": RandomForestClassifier(
            n_estimators=100,
            max_depth=14,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),
        "preprocessor_args": dict(scale_numeric=False, dense_output=False, include_categorical=True),
    },
    "Gradient Boosting": {
        "model": HistGradientBoostingClassifier(
            max_depth=8,
            max_iter=120,
            learning_rate=0.1,
            random_state=42
        ),
        "preprocessor_args": dict(scale_numeric=False, dense_output=False, include_categorical=False),
    },
    "Deep Neural Network": {
        "model": MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            max_iter=30,
            early_stopping=True,
            random_state=42
        ),
        "preprocessor_args": dict(scale_numeric=True, dense_output=False, include_categorical=True),
    },
}

In [ ]:
results = []
trained_pipelines = {}

for model_name, config in models.items():
    print(f"Training: {model_name}")
    preprocessor = build_preprocessor(**config["preprocessor_args"])
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", config["model"]),
    ])

    pipeline.fit(X_train, y_train)
    trained_pipelines[model_name] = pipeline

    y_pred = pipeline.predict(X_test)

    if hasattr(pipeline, "predict_proba"):
        y_score = pipeline.predict_proba(X_test)[:, 1]
    elif hasattr(pipeline, "decision_function"):
        y_score = pipeline.decision_function(X_test)
    else:
        y_score = y_pred

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F2 Score": fbeta_score(y_test, y_pred, beta=2, zero_division=0),
        "F2 Macro": fbeta_score(y_test, y_pred, beta=2, average="macro", zero_division=0),
        "PR AUC": average_precision_score(y_test, y_score),
    })

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title(f"Confusion Matrix - {model_name}")
    plt.tight_layout()
    plt.show()

    try:
        plt.figure(figsize=(5, 4))
        PrecisionRecallDisplay.from_predictions(y_test, y_score)
        plt.title(f"Precision-Recall Curve - {model_name}")
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Could not plot PR curve for {model_name}: {e}")

In [ ]:
results_df = pd.DataFrame(results).sort_values(
    by=["F2 Score", "Recall", "PR AUC"],
    ascending=False
).reset_index(drop=True)

results_df

## Final Discussion Points
Use these points in your report:
1. The test set has a different class balance from the training set, so this is a realistic intrusion-detection setting with distribution shift.
2. Recall and F2 are especially important because missing attacks is usually more dangerous than raising extra alarms.
3. Tree-based and ensemble models often handle nonlinear traffic behavior better, while linear models are easier to interpret.
4. Scaling improved distance-based models such as KNN and margin-based models such as SVM.
5. Continuous monitoring is required in real deployments because traffic behavior changes over time.